# Reasoning Loops — Clear States and Hard Limits

Based on:
- [Language models can overthink](https://the-decoder.com/language-models-can-overthink-and-get-stuck-in-endless-thought-loops/) — The Decoder, Jan 2025
- [How many reasoning steps do AI agents need](https://particula.tech/blog/ai-agent-loops-reasoning-steps-optimization) — Particula, Jul 2025
- [How to Prevent Infinite Loops](https://codieshub.com/for-ai/prevent-agent-loops-costs) — CodiesHub, Dec 2025

## The Problem

Agents loop when tools give ambiguous feedback. If a tool always says "prices may change — try again", the agent has no signal to stop. It retries the same call with the same input, burning tokens with each iteration, until it hits the model's iteration limit.

This is not a bug in the tool. It's a design choice that accidentally removes the stopping condition.

## The Tools

Four tools in `tools.py` — two that **cause** loops, two that **prevent** them:

| Tool | Returns | Effect on agent |
|------|---------|-----------------|
| `search_flights(origin, destination, max_price)` | `"More results may be available. Prices change frequently."` | ❌ Ambiguous — agent retries hoping for better prices |
| `check_hotel_price(hotel, check_in)` | `"Prices may change — check again for latest."` | ❌ Ambiguous — agent retries to get current price |
| `book_flight(flight, passenger)` | `"SUCCESS: FL12345"` or `"FAILED: No seats"` | ✅ Clear — agent stops on SUCCESS |
| `book_hotel(hotel, guest, nights)` | `"SUCCESS: HT67890 confirmed"` or `"FAILED: Fully booked"` | ✅ Clear — agent stops on SUCCESS |

## The Hook

One hook in `hooks.py` intercepts tool calls via `BeforeToolCallEvent`:

**`LimitToolCounts(max_tool_counts)`** — the official recipe from the [Strands Hooks Cookbook](https://strandsagents.com/docs/user-guide/concepts/agents/hooks/). A hard ceiling per tool per invocation. When a tool exceeds its limit, subsequent calls are cancelled via `event.cancel_tool`. The agent cannot exceed the ceiling regardless of LLM behavior.

![Ambiguous Tool Feedback vs Clear States + Hard Limits](../images/Ambiguous-Tool-Feedback.png)

## What We Test

| Test | Tools | Hook | Query |
|------|-------|------|-------|
| 1 — Ambiguous (problem) | `search_flights`, `check_hotel_price` | None | Budget query |
| 2 — Clear states | `book_flight`, `book_hotel` | None | Booking query |
| 3 — Hard limits | `search_flights`, `check_hotel_price` | `LimitToolCounts(max=3)` | Multi-city query |

## 📦 Setup

In [1]:
import os
import time
os.environ['OTEL_SDK_DISABLED'] = 'true'
import logging, warnings  # silence OpenTelemetry 'Failed to detach context' noise
logging.getLogger('opentelemetry').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore', message='Failed to detach context')

from dotenv import load_dotenv
from strands import Agent
from strands.models.openai import OpenAIModel
from tools import search_flights, check_hotel_price, book_flight, book_hotel
from hooks import LimitToolCounts

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("⚠️ OPENAI_API_KEY not set. Get yours at https://platform.openai.com/api-keys and add OPENAI_API_KEY=your-key to a .env file.")

MODEL = OpenAIModel(model_id="gpt-4o-mini")

# ─────────────────────────────────────────────────────────────────────────────
# How to switch the model provider (token counting works the same on all of them).
#
# Amazon Bedrock — uses boto3, NO extra package needed.
#   Requires configured AWS credentials (e.g. `aws configure` or environment variables)
#   and model access enabled in the Amazon Bedrock console.
#       from strands.models import BedrockModel
#       MODEL = BedrockModel(
#           model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
#           region_name="us-east-1",
#       )
#   Docs: https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/
#
# Anthropic (direct API) — requires the extra:  pip install 'strands-agents[anthropic]'
#   The API key goes inside client_args (get one at https://console.anthropic.com/).
#       from strands.models.anthropic import AnthropicModel
#       MODEL = AnthropicModel(
#           client_args={"api_key": os.getenv("ANTHROPIC_API_KEY")},
#           model_id="claude-sonnet-4-6",
#           max_tokens=1028,
#       )
#   Docs: https://strandsagents.com/docs/user-guide/concepts/model-providers/anthropic/
# ─────────────────────────────────────────────────────────────────────────────

# Prompt that causes the agent to retry when it can't find prices within budget —
# this is what triggers organic loops in Scenario 1.
PERSISTENT_PROMPT = (
    "You are a persistent travel agent. Always try to find prices within the user's budget. "
    "If results are over budget, search again — prices fluctuate and you might find better deals on retry."
)

# Query that triggers the organic loop in Scenario 1 (ambiguous feedback, no ceiling).
BUDGET_QUERY = "Find me the cheapest flight from NYC to Paris under $400 and a hotel under $200/night for March 15"

def count_tool_calls(agent):
    count = 0
    for msg in agent.messages:
        for block in msg.get("content", []):
            if "toolUse" in block:
                count += 1
    return count

print("✅ Setup complete!")

✅ Setup complete!


---

## 🔬 Scenario 1: Baseline (No Loop Detection)

**Research:** "Agent calls same tool repeatedly without progress"

**Expected:** May make redundant calls without detection

In [2]:
agent_loop = Agent(
    model=MODEL,
    system_prompt=PERSISTENT_PROMPT,
    tools=[search_flights, check_hotel_price],
)

start = time.time()
response = agent_loop(BUDGET_QUERY)
time_loop = time.time() - start
calls_loop = count_tool_calls(agent_loop)

print(f"⏱️  {time_loop:.1f}s — {calls_loop} tool calls")
if calls_loop > 4:
    print(f"⚠️  {calls_loop} calls — ambiguous feedback caused retries")
else:
    print("ℹ️  Agent stopped early (LLM behavior varies run-to-run)")
# 📊 Token counting — Strands native metric (same for OpenAI, Bedrock, etc.).
# result.metrics.accumulated_usage sums ALL the LLM calls made during this task:
#   inputTokens  = tokens sent (prompt + system prompt + tool definitions + history)
#   outputTokens = tokens generated by the model
#   totalTokens  = input + output
# We check '.metrics' because it can be None if the provider does not report usage.
tokens_loop = response.metrics.accumulated_usage['totalTokens'] if response.metrics else 0
print(f"💰 Tokens: {tokens_loop:,} total")



Tool #1: search_flights

Tool #2: check_hotel_price



Tool #3: search_flights

Tool #4: check_hotel_price



Tool #5: search_flights

Tool #6: check_hotel_price



Tool #7: search_flights

Tool #8: check_hotel_price


I found some options for your trip from

 NYC to Paris.

### Flights:
1. **AirLine B:** $230
2

. **AirLine A:** $347
3. **AirLine C:** $341

### Hotels:
- **

Another Budget Hotel Paris:** $185/night (selling fast)

Both options

 for flights are under $400, and the hotel is under $200/night. Would you like to proceed with booking any of these

, or do you need more information?⏱️  8.8s — 8 tool calls
⚠️  8 calls — ambiguous feedback caused retries
💰 Tokens: 2,876 total


---

## ✅ Scenario 2: Clear Success States

**Research:** "Tools return SUCCESS/FAILED, agent knows when to stop"

**Expected:** Agent stops after receiving SUCCESS

In [3]:
agent_clear = Agent(
    model=MODEL,
    tools=[book_flight, book_hotel],
)

query_book = "Book a flight NYC to Paris for Alex Rivera, and a hotel called Le Marais for 3 nights"

start = time.time()
response = agent_clear(query_book)
time_clear = time.time() - start
calls_clear = count_tool_calls(agent_clear)

print(f"⏱️  {time_clear:.1f}s — {calls_clear} tool calls")
print("✅ SUCCESS states — agent stopped immediately")
# Tokens for this call (Strands native metric, see explanation above).
tokens_clear = response.metrics.accumulated_usage['totalTokens'] if response.metrics else 0
print(f"💰 Tokens: {tokens_clear:,} total")



Tool #1: book_flight

Tool #2: book_hotel


Your bookings are confirmed:

- **Flight**: Alex Rivera is booked on flight FL88871 from NYC to Paris.
- **

Hotel**: Alex Rivera has a reservation at Le Marais for 3 nights, with a total cost of

 $684. 

Safe travels!⏱️  2.4s — 2 tool calls
✅ SUCCESS states — agent stopped immediately
💰 Tokens: 514 total


---

## 🔢 Scenario 3: Hard Limits

**Research:** "Treat every agent run as bounded process with explicit limits"

**Expected:** Agent stops at reasonable iteration count

In [4]:
limit_hook = LimitToolCounts(max_tool_counts={
    "search_flights": 2,
    "check_hotel_price": 2,
})

agent_limits = Agent(
    model=MODEL,
    system_prompt="You are a travel agent. Find the best deal for the user.",
    tools=[search_flights, check_hotel_price],
    hooks=[limit_hook],
)

query_multi = "Compare flights and hotels for NYC to Paris, London, and Tokyo — find the cheapest option for each"

start = time.time()
response = agent_limits(query_multi)
time_limits = time.time() - start
calls_limits = sum(limit_hook.tool_counts.values())

print(f"⏱️  {time_limits:.1f}s — tool counts: {limit_hook.tool_counts}")
print("✅ Hard ceiling enforced")
# Tokens for this call (Strands native metric, see explanation above).
tokens_limits = response.metrics.accumulated_usage['totalTokens'] if response.metrics else 0
print(f"💰 Tokens: {tokens_limits:,} total")



Tool #1: search_flights

Tool #2: search_flights

Tool #3: search_flights

Tool #4: check_hotel_price

Tool #5: check_hotel_price

Tool #6: check_hotel_price
🚫 Limit reached! search_flights blocked after 2 calls
🚫 Limit reached! check_hotel_price blocked after 2 calls



Tool #7: check_hotel_price

Tool #8: search_flights
🚫 Limit reached! check_hotel_price blocked after 2 calls
🚫 Limit reached! search_flights blocked after 2 calls


Here are the best flight and hotel options for your trip from NYC to Paris,

 London, and Tokyo:

### Paris


- **Cheapest Flight**: Air

...

 for **$372/night** (limited availability)

### Tokyo
- **Che

apest Flight**: Unfortunately, I couldn't retrieve flight options as the search hit the

 limit.
- **Hotel**: The Peninsula Tokyo price could not be retrieved due to the tool limit.

Would

 you like me to try again for the flights to Tokyo or

 provide additional assistance?⏱️  8.7s — tool counts: {'search_flights': 4, 'check_hotel_price': 4}
✅ Hard ceiling enforced
💰 Tokens: 1,803 total


---
## Summary

**Strands Agents makes loop prevention simple**: attach a `HookProvider` to `Agent(hooks=[...])` and Strands intercepts every tool call via `BeforeToolCallEvent` — no external libraries, no agent modification, no custom loop detection code.

```python
# All it takes:
agent = Agent(tools=[search_flights, check_hotel_price], hooks=[LimitToolCounts(max_tool_counts={"search_flights": 3})])
```

### When to Use Each Solution

| Solution | Best for |
|----------|----------|
| **Clear SUCCESS/FAILED states** | Tools you control — design unambiguous terminal states |
| **LimitToolCounts** (Strands Cookbook) | Hard cost ceilings — non-negotiable regardless of LLM behavior |

### Next Steps

1. ➡️ [Demo 01: Context Overflow](../01-context-overflow-demo/) — Memory Pointer Pattern
2. ➡️ [Demo 02: MCP Timeout](../02-mcp-timeout-demo/) — Async handleId pattern

### References

- [Language models can overthink](https://the-decoder.com/language-models-can-overthink-and-get-stuck-in-endless-thought-loops/) — The Decoder, Jan 2025
- [How many reasoning steps do AI agents need](https://particula.tech/blog/ai-agent-loops-reasoning-steps-optimization) — Particula, Jul 2025

### Strands Agents

- [Strands Hooks](https://strandsagents.com/docs/user-guide/concepts/agents/hooks/) — `BeforeToolCallEvent`, `cancel_tool`, `HookProvider`
- [Strands Hooks Cookbook](https://strandsagents.com/docs/user-guide/concepts/agents/hooks/) — `LimitToolCounts` and other patterns
- [Strands Model Providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/) — Swap to Amazon Bedrock, Anthropic, Ollama
- [Strands Agents Documentation](https://strandsagents.com) — Full framework docs
- [Code Repository](https://github.com/aws-samples/sample-why-agents-fail)